<a href="https://colab.research.google.com/github/truonghoanganh2709/bigdata/blob/main/Instruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!uname -a

Linux dfc5636e7ca1 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux


In [3]:
!cat /etc/os-release | head

PRETTY_NAME="Ubuntu 24.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.5 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"


In [4]:
!apt-get update -qq
!apt-get install -y openjdk-21-jdk-headless

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-21-jdk-headless is already the newest version (21.0.12.1+1-1~24.04.4).
0 upgraded, 0 newly installed, 0 to remove and 10 not upgraded.


In [5]:
!wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz -O /content/hadoop-3.5.0.tar.gz !ls -lh /content/hadoop-3.5.0.tar.gz

^C


In [6]:
!ls -lh /content/hadoop-3.5.0.tar.gz

-rw-r--r-- 1 root root 44M Oct  7 10:21 /content/hadoop-3.5.0.tar.gz


In [7]:
!rm -f /content/hadoop-3.5.0.tar.gz

In [8]:
!wget https://dlcdn.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz -O /content/hadoop-3.5.0.tar.gz

--2026-10-07 10:22:16--  https://dlcdn.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz
Resolving dlcdn.apache.org (dlcdn.apache.org)... 151.101.2.132, 2a04:4e42::644
Connecting to dlcdn.apache.org (dlcdn.apache.org)|151.101.2.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 581280703 (554M) [application/x-gzip]
Saving to: ‘/content/hadoop-3.5.0.tar.gz’

/content/hadoop-3.5 100%[===================>] 554.35M  82.7MB/s    in 8.5s    

2026-10-07 10:22:35 (65.2 MB/s) - ‘/content/hadoop-3.5.0.tar.gz’ saved [581280703/581280703]



In [9]:
!ls -lh /content/hadoop-3.5.0.tar.gz

-rw-r--r-- 1 root root 555M Apr  1  2026 /content/hadoop-3.5.0.tar.gz


In [10]:
!gzip -t /content/hadoop-3.5.0.tar.gz && echo "FILE OK"

FILE OK


In [11]:
!tar -xzf /content/hadoop-3.5.0.tar.gz -C /content

In [12]:
import os
import subprocess

java_home = subprocess.check_output(
    "dirname $(dirname $(readlink -f $(which java)))",
    shell=True,
    text=True
).strip()

os.environ["JAVA_HOME"] = java_home
os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["PATH"] += ":/content/hadoop-3.5.0/bin:/content/hadoop-3.5.0/sbin"

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])

JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
HADOOP_HOME = /content/hadoop-3.5.0


In [13]:
import os

hadoop_env = "/content/hadoop-3.5.0/etc/hadoop/hadoop-env.sh"

with open(hadoop_env, "a") as f:
    f.write(f"\nexport JAVA_HOME={os.environ['JAVA_HOME']}\n")

print("Đã cập nhật", hadoop_env)

Đã cập nhật /content/hadoop-3.5.0/etc/hadoop/hadoop-env.sh


In [14]:
!hadoop version

[0.027s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.027s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


In [15]:
!mkdir -p /content/hadoop_data/namenode
!mkdir -p /content/hadoop_data/datanode
!ls -ld /content/hadoop_data/*

drwxr-xr-x 2 root root 4096 Oct  7 10:26 /content/hadoop_data/datanode
drwxr-xr-x 2 root root 4096 Oct  7 10:26 /content/hadoop_data/namenode


In [16]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/core-site.xml
<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>

<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/core-site.xml


In [17]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml
<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>

<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>

    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:///content/hadoop_data/namenode</value>
    </property>

    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:///content/hadoop_data/datanode</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml


In [18]:
%%writefile /content/hadoop-3.5.0/etc/hadoop/mapred-site.xml
<?xml version="1.0"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>

<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>local</value>
    </property>
</configuration>

Overwriting /content/hadoop-3.5.0/etc/hadoop/mapred-site.xml


In [19]:
!hdfs namenode -format -force

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-07 10:29:20,725 INFO namenode.NameNode: STARTUP_MSG: 
/************************************************************
STARTUP_MSG: Starting NameNode
STARTUP_MSG:   host = dfc5636e7ca1/172.28.0.12
STARTUP_MSG:   args = [-format, -force]
STARTUP_MSG:   version = 3.5.0
STARTUP_MSG:   classpath = /content/hadoop-3.5.0/etc/hadoop:/content/hadoop-3.5.0/share/hadoop/common/lib/kerb-crypto-2.0.3.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/commons-math3-3.6.1.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/guava-33.4.8-jre.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/jetty-util-ajax-9.4.58.v20250814.jar:/content/hadoop-3.5.0/share/hadoop/common/lib/nett

In [20]:
!hdfs --daemon start namenode

In [21]:
!hdfs --daemon start datanode

In [22]:
!jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
10370 Jps
10219 DataNode
10047 NameNode


In [23]:
!hdfs dfsadmin -report

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Configured Capacity: 115658190848 (107.72 GB)
Present Capacity: 91518836736 (85.23 GB)
DFS Remaining: 91518812160 (85.23 GB)
DFS Used: 24576 (24 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Badly Distributed Blocks: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Badly Distributed Blocks: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion block

In [24]:
!hdfs dfs -ls /

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [25]:
!hdfs dfs -mkdir -p /user/student/input
!hdfs dfs -mkdir -p /user/student/output
!hdfs dfs -ls -R /user/student

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
drwxr-xr-x   

In [26]:
!printf "Hanoi,1200\nDanang,850\nHCM,1700\nHanoi,950\n" > /content/sales.txt
!cat /content/sales.txt

Hanoi,1200
Danang,850
HCM,1700
Hanoi,950


In [27]:
!hdfs dfs -put /content/sales.txt /user/student/input/

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [28]:
!hdfs dfs -ls -h /user/student/input

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 1 items
-rw-r--r--   1 root supergroup         41 2026-10-07 10:32 /user/student/input/sales.txt


In [29]:
!hdfs dfs -cat /user/student/input/sales.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950


In [30]:
!hdfs dfs -head /user/student/input/sales.txt
!hdfs dfs -tail /user/student/input/sales.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950


In [31]:
!hdfs dfs -cp /user/student/input/sales.txt /user/student/input/sales_backup.txt
!hdfs dfs -ls /user/student/input

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 2 items
-rw-r--r--   1 root supergroup         41 2026-10-07 10:32 /user/student/input/sales.txt
-rw-r--r--   1 root supergroup         41 2026-10-07 10:34 /user/student/input/sales_backup.txt


In [32]:
!hdfs dfs -mv /user/student/input/sales_backup.txt /user/student/input/sales_2026.txt
!hdfs dfs -ls /user/student/input

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 2 items
-rw-r--r--   1 root supergroup         41 2026-10-07 10:32 /user/student/input/sales.txt
-rw-r--r--   1 root supergroup         41 2026-10-07 10:34 /user/student/input/sales_2026.txt


In [33]:
!hdfs dfs -du -h /user/student/input
!hdfs dfs -df -h /

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
41  41  /user/student/input/sales.txt
41  41  /user/student/input/sales_2026.txt
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Filesystem                Size    Used  Available  Use%
hdfs://localhost:9000  107.7 G  24.1 K     85.2 G    0%


In [34]:
!rm -f /content/sales_downloaded.txt
!hdfs dfs -get /user/student/input/sales.txt /content/sales_downloaded.txt
!cat /content/sales_downloaded.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hanoi,1200
Danang,850
HCM,1700
Hanoi,950


In [35]:
!hdfs dfs -rm /user/student/input/sales_2026.txt
!hdfs dfs -ls /user/student/input

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Deleted /user/student/input/sales_2026.txt
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 1 items
-rw-r--r--   1 root supergroup         41 2026-10-07 10:32 /user/student/input/sales.txt


In [36]:
!hdfs fsck /user/student/input/sales.txt -files -blocks -locations

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Connecting to namenode via http://localhost:9870/fsck?ugi=root&files=1&blocks=1&locations=1&path=%2Fuser%2Fstudent%2Finput%2Fsales.txt
FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path /user/student/input/sales.txt at Wed Oct 07 10:35:32 UTC 2026

/user/student/input/sales.txt 41 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1663936942-172.28.0.12-1791368962690:blk_1073741825_1001 len=41 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-3fb1def7-94c3-4598-b6d2-6cb7e5359497,DISK]]


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	41 B
 Total files:	1
 Tota

In [37]:
!hdfs fsck /

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Connecting to namenode via http://localhost:9870/fsck?ugi=root&path=%2F
FSCK started by root (auth:SIMPLE) from /127.0.0.1 for path / at Wed Oct 07 10:35:50 UTC 2026


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			5
 Total symlinks:		0

Replicated Blocks:
 Total size:	41 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 41 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-replicated blocks:	0 (0.0 %)
 Mis-replicated blocks:		0 (0.0 %)
 Default replication factor:	1
 Average block replication:	1.0
 Missing blocks:		0
 Corrupt blocks:		0
 Missing replicas:		0 (0.0 %)
 Blocks queued fo

In [38]:
!hdfs dfsadmin -report

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Configured Capacity: 115658190848 (107.72 GB)
Present Capacity: 91518808116 (85.23 GB)
DFS Remaining: 91518783488 (85.23 GB)
DFS Used: 24628 (24.05 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Badly Distributed Blocks: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Badly Distributed Blocks: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion bl

In [39]:
!printf "hadoop big data hadoop mapreduce\nbig data analytics hadoop\n" > /content/words.txt
!hdfs dfs -mkdir -p /user/student/wordcount/input
!hdfs dfs -put -f /content/words.txt /user/student/wordcount/input/
!hdfs dfs -cat /user/student/wordcount/input/words.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
hadoop big da

In [40]:
!hdfs dfs -rm -r -f /user/student/wordcount/output

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [41]:
!hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar wordcount /user/student/wordcount/input /user/student/wordcount/output

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-07 10:37:24,806 INFO input.FileInputFormat: Total input files to process : 1
2026-10-07 10:37:24,856 INFO mapreduce.JobSubmitter: number of splits:1
2026-10-07 10:37:25,290 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local191020240_0001
2026-10-07 10:37:25,292 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-07 10:37:25,592 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-10-07 10:37:25,593 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-10-07 10:37:25,594 INFO mapreduce.Job: Running job: job_local191020240_0001
2026-10-07 10:37:25,610 INFO output.PathOutputCommitterFactory: No out

In [42]:
!hdfs dfs -cat /user/student/wordcount/output/part-r-00000

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
analytics	1
big	2
data	2
hadoop	3
mapreduce	1


In [43]:
!hdfs --daemon stop datanode
!hdfs --daemon stop namenode
!jps

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
13639 Jps


In [44]:
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!jps
!hdfs dfs -ls /user/student

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
13860 Jps
13813 DataNode
13751 NameNode
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ls: Call From dfc5636e7ca1/172.28.0.12 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused


In [45]:
[warning][os,container] Cgroup memory controller path ... detected limits won't be accurate
[warning][os,container] Cgroup cpu controller path ... detected limits won't be accurate

SyntaxError: unterminated string literal (detected at line 1) (1080207790.py, line 1)

In [46]:
java.net.ConnectException: Call From ... to localhost:9000
failed java.net.ConnectException: Connection refused

SyntaxError: invalid syntax (2863666958.py, line 1)